<a href="https://colab.research.google.com/github/SheffieldML/GPc/blob/main/python/examples/weightspace_ridge_1d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Weight-space ridges vs point inducing (MLP)

CIP-0028 via the `gpc` façade. Toy 1-D regression with DTCVAR: point inducing in input space versus ridge windows in whitened weight space.

```bash
pip install matplotlib   # or: pip install 'gpc[plot]'
```


In [ ]:
%pip install --no-cache-dir --upgrade --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "gpc==0.1.0a2"


In [ ]:
%pip install matplotlib


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import sys
from pathlib import Path

root = Path.cwd()
if (root / "gpc").is_dir():
    sys.path.insert(0, str(root))
elif (root.parent / "gpc").is_dir():
    sys.path.insert(0, str(root.parent))

import numpy as np
import gpc

rng = np.random.default_rng(0)
X = np.linspace(-1.5, 1.5, 80)[:, None]
Y = np.sin(2.0 * X) + 0.1 * np.cos(3.0 * X) + 0.05 * rng.normal(size=X.shape)
Z = np.linspace(-1.2, 1.2, 12)[:, None]

m_pts = gpc.SparseGPRegression(X, Y, Z, kernel="mlp", inducing="points")
m_ws = gpc.SparseGPRegression(
    X, Y, Z, kernel="mlp", inducing="weightspace", window_width=0.55
)
for m in (m_pts, m_ws):
    m.optimize(iters=80)

print("points objective:", m_pts.objective(), "inducing:", m_pts.inducing_type)
print("weightspace objective:", m_ws.objective(), "s=", m_ws.window_width)


In [ ]:
from gpc.plot import plot_1d
import matplotlib.pyplot as plt

Xtest = np.linspace(-1.8, 1.8, 300)[:, None]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
plot_1d(m_pts, X, Y, Xtest=Xtest, ax=axes[0], label="points")
axes[0].set_title("MLP + point inducing")
plot_1d(m_ws, X, Y, Xtest=Xtest, ax=axes[1], label="weightspace")
axes[1].set_title("MLP + weight-space ridges")
fig.suptitle("DTCVAR — same M, different inducing geometry")
plt.tight_layout()
plt.show()
